In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import \
    RandomForestClassifier
from sklearn.model_selection import \
    StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    silhouette_score,
    adjusted_rand_score)
from sklearn.mixture import \
    GaussianMixture
from statsmodels.stats.multitest import \
    multipletests
import os
import json

DATA_DIR   = '/rds/homes/j/jxt554/data'
TABLES_DIR = '/rds/homes/j/jxt554/tables'
SEED       = 42
np.random.seed(SEED)

print('IBDome UC Pipeline')
print('='*55)

# ── Check available UC files ──────────
print('Checking UC files...')
uc_files = [
    'ibdome_uc_preprocessed.csv',
    'ibdome_uc_meta.csv',
    'ibdome_uc_final.csv',
    'ibdome_uc_labels_dec_consensus.npy',
    'ibdome_uc_consensus_dec.npy',
    'ibdome_uc_stability_dec.npy',
    'ibdome_uc_latent_dae_dec.npy',
]
for f in uc_files:
    path = f'{DATA_DIR}/{f}'
    if os.path.exists(path):
        if f.endswith('.npy'):
            arr = np.load(path)
            print(f'  FOUND {f}: {arr.shape}')
        else:
            df = pd.read_csv(path, nrows=2)
            print(f'  FOUND {f}: '
                   f'{df.shape}')
    else:
        print(f'  MISSING {f}')

IBDome UC Pipeline
Checking UC files...
  FOUND ibdome_uc_preprocessed.csv: (2, 61)
  FOUND ibdome_uc_meta.csv: (2, 116)
  FOUND ibdome_uc_final.csv: (2, 120)
  FOUND ibdome_uc_labels_dec_consensus.npy: (132,)
  FOUND ibdome_uc_consensus_dec.npy: (132, 132)
  FOUND ibdome_uc_stability_dec.npy: (132,)
  FOUND ibdome_uc_latent_dae_dec.npy: (132, 16)


In [3]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import \
    RandomForestClassifier
from sklearn.model_selection import \
    StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    silhouette_score,
    adjusted_rand_score)
from sklearn.mixture import \
    GaussianMixture
from statsmodels.stats.multitest import \
    multipletests
import os
import json

DATA_DIR   = '/rds/homes/j/jxt554/data'
TABLES_DIR = '/rds/homes/j/jxt554/tables'
SEED       = 42
np.random.seed(SEED)

print('IBDome UC — Original Setup')
print('='*55)

# ── Load EXACT original UC files ──────
X_int = pd.read_csv(
    f'{DATA_DIR}/'
    'ibdome_uc_preprocessed.csv')
prot_cols = X_int.columns.tolist()
X_int     = X_int.values

meta = pd.read_csv(
    f'{DATA_DIR}/ibdome_uc_final.csv')

labels_consensus = np.load(
    f'{DATA_DIR}/'
    'ibdome_uc_labels_dec_consensus.npy')
C_mat = np.load(
    f'{DATA_DIR}/'
    'ibdome_uc_consensus_dec.npy')
stab_orig = np.load(
    f'{DATA_DIR}/'
    'ibdome_uc_stability_dec.npy')

N = len(X_int)
P = len(prot_cols)

print(f'X shape     : {X_int.shape}')
print(f'Labels      : '
       f'{np.bincount(labels_consensus)}')
print(f'N match     : '
       f'{N == len(labels_consensus)}')

# ── Metrics ───────────────────────────
D_mat = np.clip(1-C_mat, 0, None)
np.fill_diagonal(D_mat, 0)
sil_final  = silhouette_score(
    D_mat, labels_consensus,
    metric='precomputed')
stab_mean  = stab_orig.mean()
n_unstable = (stab_orig < 0.5).sum()
cluster_sizes = np.bincount(
    labels_consensus)

print(f'\nClustering:')
print(f'  Silhouette : {sil_final:.4f}')
print(f'  Stability  : {stab_mean:.4f}')
print(f'  Unstable   : {n_unstable}')
print(f'  Sizes      : '
       f'{cluster_sizes.tolist()}')

# ── Identify hyperinflam ──────────────
inflam_prots = [
    'TNF','IL6','IL8','IL10',
    'HGF','OSM','CXCL10',
    'CXCL9','IFN-gamma','CCL4',
    'MCP-1','MCP-3','VEGFA',
    'MMP-10','MMP-1']
found_idx = [
    prot_cols.index(p)
    for p in inflam_prots
    if p in prot_cols]

sc0 = X_int[
    labels_consensus==0][:,
    found_idx].mean()
sc1 = X_int[
    labels_consensus==1][:,
    found_idx].mean()

print(f'\nInflammatory scores:')
print(f'  C0 (n={cluster_sizes[0]}): '
       f'{sc0:.4f}')
print(f'  C1 (n={cluster_sizes[1]}): '
       f'{sc1:.4f}')

hyper_label = 0 if sc0>sc1 else 1
quiet_label = 1 - hyper_label
print(f'  Hyperinflam: '
       f'C{hyper_label+1} '
       f'(n={cluster_sizes[hyper_label]})')
print(f'  Quiescent  : '
       f'C{quiet_label+1} '
       f'(n={cluster_sizes[quiet_label]})')

# Check MMP-10 — top UC protein
for prot in ['MMP-10','HGF','TNF']:
    if prot in prot_cols:
        pi = prot_cols.index(prot)
        h  = [X_int[labels_consensus==c,
                      pi].mean()
               for c in range(2)]
        print(f'  {prot}: '
               f'C0={h[0]:.4f} '
               f'C1={h[1]:.4f}')

# ── DE with Welch t-test ──────────────
print('\nDifferential abundance...')
de_results = []
X1 = X_int[labels_consensus==0]
X2 = X_int[labels_consensus==1]

for j, prot in enumerate(prot_cols):
    t, p = stats.ttest_ind(
        X1[:,j], X2[:,j],
        equal_var=False)
    fc = X2[:,j].mean() - X1[:,j].mean()
    de_results.append({
        'protein' : prot,
        'logFC'   : round(fc, 5),
        't_stat'  : round(t, 5),
        'p_value' : p,
        'mean_C1' : round(
            X1[:,j].mean(), 5),
        'mean_C2' : round(
            X2[:,j].mean(), 5),
    })

de_df = pd.DataFrame(de_results)
_, fdr, _, _ = multipletests(
    de_df['p_value'],
    method='fdr_bh')
de_df['adj_p_value'] = fdr
de_df['significant'] = fdr < 0.05
de_df = de_df.sort_values('adj_p_value')

n_sig   = de_df['significant'].sum()
n_up_c2 = ((de_df['significant']) &
            (de_df['logFC']>0)).sum()
n_up_c1 = ((de_df['significant']) &
            (de_df['logFC']<0)).sum()

print(f'  Significant : {n_sig}')
print(f'  Higher C2   : {n_up_c2}')
print(f'  Higher C1   : {n_up_c1}')
print(f'\n  Top 10:')
for _, r in de_df.head(10).iterrows():
    sig = 'sig' if r['significant'] \
        else 'ns'
    print(f'    {r["protein"]:<15} '
           f'logFC={r["logFC"]:+.3f} '
           f'FDR={r["adj_p_value"]:.2e} '
           f'{sig}')

de_df.to_csv(
    f'{TABLES_DIR}/'
    'ibdome_uc_de_v2.csv',
    index=False)
print('  DE saved')

# ── RF ────────────────────────────────
print('\nRandom Forest...')
y_rf = (labels_consensus ==
        hyper_label).astype(int)

rf   = RandomForestClassifier(
    n_estimators=500,
    max_features='sqrt',
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1,
    class_weight='balanced')
cv   = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED)
aucs = []

for fold,(tr,te) in enumerate(
        cv.split(X_int, y_rf)):
    rf.fit(X_int[tr], y_rf[tr])
    prob = rf.predict_proba(
        X_int[te])[:,1]
    auc  = roc_auc_score(
        y_rf[te], prob)
    aucs.append(auc)
    print(f'  Fold {fold+1}: '
           f'AUC={auc:.4f}')

mean_auc = np.mean(aucs)
std_auc  = np.std(aucs)
print(f'\n  CV AUC = {mean_auc:.4f} '
       f'+- {std_auc:.4f}')

rf.fit(X_int, y_rf)
rf_df = pd.DataFrame({
    'protein'  : prot_cols,
    'gini_imp' : rf.feature_importances_,
}).sort_values(
    'gini_imp',
    ascending=False
).reset_index(drop=True)

de_sig_set = set(
    de_df[de_df['significant']][
        'protein'].tolist())
top20 = set(
    rf_df.head(20)['protein'].tolist())
top30 = set(
    rf_df.head(
        min(30, len(rf_df))
    )['protein'].tolist())
overlap20 = top20 & de_sig_set
overlap30 = top30 & de_sig_set

print(f'  RF-DE top20: '
       f'{len(overlap20)}/20 '
       f'({len(overlap20)/20*100:.0f}%)')
print(f'  RF-DE top30: '
       f'{len(overlap30)}/30 '
       f'({len(overlap30)/30*100:.0f}%)')

rf_df.to_csv(
    f'{TABLES_DIR}/'
    'ibdome_uc_rf_v2.csv',
    index=False)
print('  RF saved')

# ── GMM ───────────────────────────────
print('\nGMM validation...')
Z_best = np.load(
    f'{DATA_DIR}/'
    'ibdome_uc_latent_dae_dec.npy')
labels_gmm = GaussianMixture(
    n_components=2,
    random_state=SEED,
    n_init=10
).fit_predict(Z_best)
ari = adjusted_rand_score(
    labels_consensus, labels_gmm)
print(f'  GMM ARI: {ari:.4f}')

from scipy.stats import mannwhitneyu

# ── Clinical characterisation ─────────
print('Clinical characterisation...')
clin_vars = {
    'age'                              : 'continuous',
    'sex'                              : 'categorical',
    'preexisting_arterial_hypertension': 'categorical',
    'preexisting_diabetes_mellitus'    : 'categorical',
    'riskfactor_nicotine'              : 'categorical',
}

clin_results = []
for var, vtype in clin_vars.items():
    col = next(
        (c for c in meta.columns
         if var.lower() in c.lower()),
        None)
    if col is None:
        continue
    g0 = meta.loc[
        labels_consensus==0, col
    ].dropna()
    g1 = meta.loc[
        labels_consensus==1, col
    ].dropna()
    if len(g0)<3 or len(g1)<3:
        continue
    if vtype == 'continuous':
        _, p = mannwhitneyu(
            pd.to_numeric(
                g0, errors='coerce'
            ).dropna(),
            pd.to_numeric(
                g1, errors='coerce'
            ).dropna(),
            alternative='two-sided')
        test = 'Mann-Whitney U'
        g0n  = pd.to_numeric(
            g0, errors='coerce')
        g1n  = pd.to_numeric(
            g1, errors='coerce')
        c0v  = (f'{g0n.mean():.1f}'
                 f'+-{g0n.std():.1f}')
        c1v  = (f'{g1n.mean():.1f}'
                 f'+-{g1n.std():.1f}')
    else:
        try:
            ct = pd.crosstab(
                labels_consensus,
                meta[col])
            _, p, _, _ = \
                stats.chi2_contingency(ct)
        except Exception:
            p = 1.0
        test = 'Chi-squared'
        c0v  = str(
            g0.value_counts().to_dict())
        c1v  = str(
            g1.value_counts().to_dict())
    sig = 'YES' if p<0.05 else 'ns'
    clin_results.append({
        'Variable'   : col,
        'Test'       : test,
        'C1_value'   : c0v,
        'C2_value'   : c1v,
        'p_value'    : round(p, 5),
        'Significant': sig,
    })
    print(f'  {col:<40} '
           f'p={p:.4f} {sig}')

pd.DataFrame(clin_results).to_csv(
    f'{TABLES_DIR}/'
    'ibdome_uc_clinical_v2.csv',
    index=False)
print('  Clinical saved')

# ── Chemokine x Vascular ──────────────
print('\nChemokine x Vascular...')
CHEMOKINE_LIST = [
    'IL8','MCP-3','MCP-1','CXCL11',
    'CXCL9','CXCL1','CCL4','CCL19',
    'CXCL5','CCL3','CXCL6','CXCL10',
    'CCL28','CCL25','CCL20']
VASCULAR_LIST  = [
    'VEGFA','HGF','FGF-21','FGF-19',
    'FGF-5','LIF','ARTN','NRTN',
    'GDNF','FGF-23','Beta-NGF']
CYTOKINE_LIST  = [
    'IL6','IL-17C','IL-17A','OSM',
    'IL18','IL10','TNF','IFN-gamma']

def pathway_score(X, plist, avail):
    present = [p for p in plist
                if p in avail]
    if not present:
        return np.zeros(X.shape[0]),[]
    idx = [avail.index(p)
            for p in present]
    return X[:,idx].mean(axis=1), present

sc_chemo, used_chemo = pathway_score(
    X_int, CHEMOKINE_LIST, prot_cols)
sc_vasc,  used_vasc  = pathway_score(
    X_int, VASCULAR_LIST, prot_cols)
sc_cyto,  used_cyto  = pathway_score(
    X_int, CYTOKINE_LIST, prot_cols)

r_cv, p_cv = stats.pearsonr(
    sc_chemo, sc_vasc)
r_cc, p_cc = stats.pearsonr(
    sc_chemo, sc_cyto)

print(f'  Chemo proteins : '
       f'{len(used_chemo)}')
print(f'  Vascular prots : '
       f'{len(used_vasc)}')
print(f'  Chemo x Vasc   : '
       f'r={r_cv:.4f} p={p_cv:.4e}')
print(f'  UKB reference  : r=0.749')
print(f'  Delta          : '
       f'{abs(r_cv-0.749):.4f}')

# ── Update metadata ───────────────────
meta['cluster']   = labels_consensus
meta['stability'] = stab_orig
meta['subtype']   = [
    'Hyperinflammatory'
    if l==hyper_label else 'Quiescent'
    for l in labels_consensus]

meta.to_csv(
    f'{DATA_DIR}/'
    'ibdome_uc_final_v2.csv',
    index=False)

X_df = pd.read_csv(
    f'{DATA_DIR}/'
    'ibdome_uc_preprocessed.csv')
X_df.to_csv(
    f'{DATA_DIR}/'
    'ibdome_uc_preprocessed_v2.csv',
    index=False)

np.save(
    f'{DATA_DIR}/'
    'ibdome_uc_consensus_v2.npy',
    C_mat)
np.save(
    f'{DATA_DIR}/'
    'ibdome_uc_labels_v2.npy',
    labels_consensus)

with open(
        f'{DATA_DIR}/'
        'ibdome_uc_validation_v2.json',
        'w') as f:
    json.dump({
        'cohort'                  : 'IBDome_UC',
        'chemokine_vascular_r'    : round(r_cv,5),
        'chemokine_cytokine_r'    : round(r_cc,5),
        'ukb_reference_r'         : 0.749,
        'delta_r'                 : round(
            abs(r_cv-0.749),5),
        'n_chemo_used'            : len(used_chemo),
        'n_vasc_used'             : len(used_vasc),
        'chemokine_proteins_used' : used_chemo,
        'vascular_proteins_used'  : used_vasc,
    }, f, indent=2)

# ── Final summary ─────────────────────
print('\nFINAL SUMMARY — IBDome UC')
print('='*55)
summary = {
    'Cohort'         : 'IBDome_UC',
    'Disease'        : 'Ulcerative Colitis',
    'Country'        : 'Germany',
    'N_patients'     : N,
    'N_proteins'     : P,
    'k'              : 2,
    'Silhouette'     : round(sil_final,4),
    'Stability'      : round(stab_mean,4),
    'N_unstable'     : int(n_unstable),
    'GMM_ARI'        : round(ari,4),
    'N_hyperinflam'  : int(
        cluster_sizes[hyper_label]),
    'N_quiescent'    : int(
        cluster_sizes[quiet_label]),
    'DE_sig'         : int(n_sig),
    'DE_up_hyper'    : int(n_up_c2),
    'DE_up_quiet'    : int(n_up_c1),
    'RF_AUC'         : round(mean_auc,4),
    'RF_AUC_std'     : round(std_auc,4),
    'RF_DE_top20'    : len(overlap20),
    'RF_DE_top30'    : len(overlap30),
    'Chemo_Vasc_r'   : round(r_cv,4),
    'UKB_ref_r'      : 0.749,
    'Delta_r'        : round(
        abs(r_cv-0.749),4),
}
for k,v in summary.items():
    print(f'  {k:<20} {v}')

pd.DataFrame([summary]).to_csv(
    f'{TABLES_DIR}/'
    'ibdome_uc_summary_v2.csv',
    index=False)

print('\nAll UC files saved')

IBDome UC — Original Setup
X shape     : (132, 61)
Labels      : [54 78]
N match     : True

Clustering:
  Silhouette : 0.8271
  Stability  : 0.8817
  Unstable   : 1
  Sizes      : [54, 78]

Inflammatory scores:
  C0 (n=54): -0.6093
  C1 (n=78): 0.4218
  Hyperinflam: C2 (n=78)
  Quiescent  : C1 (n=54)
  MMP-10: C0=-0.7389 C1=0.5115
  HGF: C0=-0.6973 C1=0.4827
  TNF: C0=-0.4958 C1=0.3432

Differential abundance...
  Significant : 53
  Higher C2   : 53
  Higher C1   : 0

  Top 10:
    MMP-10          logFC=+1.250 FDR=1.28e-13 sig
    CD5             logFC=+1.238 FDR=1.28e-13 sig
    IL8             logFC=+1.208 FDR=1.59e-13 sig
    VEGFA           logFC=+1.192 FDR=1.59e-13 sig
    MCP-3           logFC=+1.226 FDR=1.59e-13 sig
    CCL20           logFC=+1.233 FDR=1.59e-13 sig
    LAP TGF-beta-1  logFC=+1.187 FDR=4.83e-13 sig
    IL-17C          logFC=+1.169 FDR=1.46e-12 sig
    HGF             logFC=+1.180 FDR=2.32e-12 sig
    TNFRSF9         logFC=+1.133 FDR=2.32e-12 sig
  DE saved

Rand

/tmp/ipykernel_3352330/497360571.py:382: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  meta['subtype']   = [
